# Bài 2 (Trang 28) - Dữ liệu Bank Marketing
## Câu 8: Tạo thuộc tính mới và chuẩn hóa Min-max

**Sinh viên:** Trần Đình Võ - **MSSV:** 6451071088

In [1]:
HO_TEN = "Trần Đình Võ"
MSSV = "6451071088"
print(f"Họ tên: {HO_TEN} - MSSV: {MSSV}")

Họ tên: Trần Đình Võ - MSSV: 6451071088


### Import thư viện và đọc dữ liệu

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

df = pd.read_csv("../Dataset/bank/bank.csv", sep=";")

### Các bước xử lý của câu 1 → 7 (chuẩn bị dữ liệu)

In [3]:
# Trong bộ dữ liệu, giá trị thiếu được ghi là 'unknown' -> chuyển về NaN
df = df.replace("unknown", np.nan)

df["subscribed"] = (df["y"] == "yes").astype(int)

df["contacts"] = df["campaign"] + df["previous"]          # tổng số lần liên hệ
df["contacted_before"] = (df["previous"] > 0).astype(int)  # đã từng được liên hệ ở chiến dịch trước

df = df.drop(columns=["poutcome", "pdays", "day"])

# job, education: điền bằng mode; contact: xem giá trị thiếu là một loại mới 'other'
df["job"] = df["job"].fillna(df["job"].mode()[0])
df["education"] = df["education"].fillna(df["education"].mode()[0])
df["contact"] = df["contact"].fillna("other")

# Thuộc tính nhị phân yes/no -> 1/0
for col in ["default", "housing", "loan", "y"]:
    df[col] = df[col].map({"no": 0, "yes": 1})
df = df.drop(columns=["subscribed"])
# Thuộc tính thứ tự
df["education"] = df["education"].map({"primary": 1, "secondary": 2, "tertiary": 3})
months = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
df["month"] = df["month"].map({m: i + 1 for i, m in enumerate(months)})
# duration (giây) -> phút (số nguyên)
df["duration_min"] = (df["duration"] / 60).astype(int)

# Thuộc tính phân loại (nominal) -> One-hot encoding (tránh quan hệ thứ bậc giả)
df = pd.get_dummies(df, columns=["job", "marital", "contact"], dtype=int)

# Tuổi -> 8 nhóm có số lượng cân bằng
df["age_group"] = pd.qcut(df["age"], 8, labels=False)
# balance -> 6 nhóm có số lượng cân bằng
df["balance_group"] = pd.qcut(df["balance"], 6, labels=False)

### Câu 8: Tạo thuộc tính mới và chuẩn hóa Min-max

In [4]:
# Tuổi * trình độ học vấn
df["age_edu"] = df["age"] * df["education"]
# Số dư trên mỗi lần liên hệ (làm trơn +1)
df["balance_per_contact"] = (df["balance"] / (df["contacts"] + 1)).astype(int)

In [5]:
df[["age", "education", "age_edu", "balance", "contacts", "balance_per_contact"]].head(10)

,age,education,age_edu,balance,contacts,balance_per_contact
0,30,1,30,1787,1,893
1,33,2,66,4789,5,798
2,35,3,105,1350,2,450
3,30,3,90,1476,4,295
4,59,2,118,0,1,0
5,35,3,105,747,5,124
6,36,3,108,307,3,76
7,39,2,78,147,2,49
8,41,3,123,221,2,73
9,43,1,43,-88,3,-22


In [6]:
# Chuẩn hóa Min-max các thuộc tính số về [0, 1]
num_cols = ["age", "balance", "duration", "campaign", "previous"]
df_norm = df.copy()
for col in num_cols:
    df_norm[col] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())

In [7]:
df_norm[num_cols].describe().round(3)

,age,balance,duration,campaign,previous
count,4521.000,4521.000,4521.000,4521.000,4521.000
mean,0.326,0.064,0.086,0.037,0.022
std,0.156,0.040,0.086,0.063,0.068
min,0.000,0.000,0.000,0.000,0.000
25%,0.206,0.045,0.033,0.000,0.000
50%,0.294,0.050,0.060,0.020,0.000
75%,0.441,0.064,0.108,0.041,0.000
max,1.000,1.000,1.000,1.000,1.000


### Nhận xét
- Tạo age_edu = age * education và balance_per_contact = balance / (contacts + 1).
- Chuẩn hóa Min-max đưa age, balance, duration, campaign, previous về [0, 1].